Используем selenium для парсинга сайта hirify

In [ ]:
!pip install selenium

In [88]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import logging
import sys

https://discourse.jupyter.org/t/jupyter-does-not-print-logging-information/11790

In [89]:
BASE_URL = "https://hirify.me"
logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s %(message)s")
logger = logging.getLogger('selenium_scrapper')
logger.setLevel(logging.DEBUG)
def prepare_driver_page(period: str = "?period=week") -> webdriver.Chrome:
    driver = webdriver.Chrome()
    logger.info("Driver created")
    driver.get(BASE_URL + period)
    time.sleep(2)
    logger.info("Page loaded")
    settings_button = driver.find_element(By.CLASS_NAME, "settings-button")
    settings_button.click()
    filter_up = driver.find_element(By.CLASS_NAME, "filters-row")
    filters = filter_up.find_elements(By.TAG_NAME, "input")
    for f in filters:
        logger.debug(f"Filter {f.get_attribute('id')} is selected: {f.is_selected()}")
        if not f.is_selected():
            f.click()
    for f in filters:
        logger.debug(f"Filter {f.get_attribute('id')} is selected: {f.is_selected()}")
    logger.info("Filters applied")
    return driver


driver = prepare_driver_page()

2026-04-07 01:57:01,369 Driver created
2026-04-07 01:57:08,174 Page loaded
2026-04-07 01:57:08,270 Filter  is selected: True
2026-04-07 01:57:08,281 Filter  is selected: False
2026-04-07 01:57:08,342 Filter  is selected: True
2026-04-07 01:57:08,352 Filter  is selected: False
2026-04-07 01:57:08,398 Filter  is selected: True
2026-04-07 01:57:08,405 Filter  is selected: True
2026-04-07 01:57:08,412 Filter  is selected: True
2026-04-07 01:57:08,419 Filter  is selected: True
2026-04-07 01:57:08,420 Filters applied


In [90]:
def parse_cards(driver: webdriver.Chrome):
    cards = driver.find_elements(By.CSS_SELECTOR, "div.vacancy-card[data-vacancy-id]")
    logger.info(f"Found {len(cards)} cards")
    return cards

cards = parse_cards(driver)

2026-04-07 01:57:08,442 Found 15 cards


In [77]:
print(len(cards))

15


In [91]:
logger.info("Parsing vacancys")

2026-04-07 01:57:11,291 Parsing vacancys


In [101]:
cards[0].text.split("\n")[5]

'Credentialing Manager'

In [ ]:
def parse_vacancys(cards: list) -> list[dict]:
    vacancys = []
    for card in cards:
        logger.info(f"Parsing card with id {card.get_attribute('data-vacancy-id')}") # Используем get_attribute на случай если у элемента нет детей
        vacancy_link = card.find_element(By.CLASS_NAME, "vacancy-card-link")
        vacancy_title = card.find_element(By.CLASS_NAME, "title")
        vacancy_tags_div = card.find_element(By.CLASS_NAME, "common-tags")
        vacancy_tags = vacancy_tags_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_tags = [t.text for t in vacancy_tags]
        try:
            vacancy_company = card.find_element(By.CLASS_NAME, "company")
        except Exception as e:
            vacancy_company = None
        vacancy_time = card.find_element(By.CLASS_NAME, "date")
        vacancy_skills_div = card.find_element(By.CLASS_NAME, "vacancy-tags")
        vacancy_skills = vacancy_skills_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_skills = [s.text for s in vacancy_skills]
        try:
            vacansy_salary = card.find_element(By.CLASS_NAME, "salary")
        except Exception as e:
            vacansy_salary = None
        vacancys.append(
            {
                "title": vacancy_title.text,
                "link": vacancy_link.get_attribute("href"),
                "tags": vacancy_tags,
                "company": vacancy_company.text if vacancy_company else "No company",
                "time": vacancy_time.text,
                "skills": vacancy_skills,
                "salary": vacansy_salary.text if vacansy_salary else "No salary",
            }
        )
    return vacancys
vacancys = parse_vacancys(cards)
print(vacancys[0])

2026-04-07 02:01:38,403 Parsing card with id 440425
2026-04-07 02:01:38,412 Card text: Credentialing Manager
2026-04-07 02:01:38,494 Parsing card with id 440423
2026-04-07 02:01:38,501 Card text: 8 minutes ago
2026-04-07 02:01:38,592 Parsing card with id 440424
2026-04-07 02:01:38,599 Card text: 8 minutes ago
2026-04-07 02:01:38,701 Parsing card with id 440422
2026-04-07 02:01:38,709 Card text: 8 minutes ago
2026-04-07 02:01:38,814 Parsing card with id 440421
2026-04-07 02:01:38,821 Card text: 8 minutes ago
2026-04-07 02:01:38,912 Parsing card with id 440420
2026-04-07 02:01:38,919 Card text: 8 minutes ago
2026-04-07 02:01:39,022 Parsing card with id 440419
2026-04-07 02:01:39,031 Card text: 8 minutes ago
2026-04-07 02:01:39,156 Parsing card with id 440418
2026-04-07 02:01:39,165 Card text: agile
2026-04-07 02:01:39,281 Parsing card with id 440417
2026-04-07 02:01:39,289 Card text: 10 minutes ago
2026-04-07 02:01:39,407 Parsing card with id 440416
2026-04-07 02:01:39,415 Card text: 10 

In [93]:
import pandas as pd

df = pd.DataFrame(vacancys)
df.head(20)

,title,link,tags,company,time,skills,salary
0,Credentialing Manager,https://hirify.me/jobs/440425-credentialing-ma...,"[onsite, US, fulltime]",Company hidden,26 seconds ago,"[compliance, credentialing, health plan enroll...",No salary
1,Software Engineer (Java),https://hirify.me/jobs/440423-software-enginee...,"[remote (Poland)/hybrid, Poland, fulltime, mid...",Company hidden,8 minutes ago,"[java, kubernetes, redis, oracle, spring, +12 ...",No salary
2,Senior Business Analyst (Fintech),https://hirify.me/jobs/440424-senior-business-...,"[onsite, Netherlands, fulltime, senior]",Company hidden,8 minutes ago,"[data analysis, finance, compliance, risk, cre...",No salary
3,Backend Engineer,https://hirify.me/jobs/440422-backend-engineer,"[onsite, UK, fulltime, middle/senior]",Company hidden,8 minutes ago,"[java, python, spring, kafka, fastapi, +3 skills]",No salary
4,Senior Software Developer,https://hirify.me/jobs/440421-senior-software-...,"[onsite, Netherlands, fulltime, senior]",Company hidden,8 minutes ago,"[c#, .net, azure, ci cd, software architecture...",No salary
5,Ai Security Specialist,https://hirify.me/jobs/440420-ai-security-spec...,"[hybrid, Netherlands, fulltime, middle/senior]",Company hidden,8 minutes ago,"[ai, ml, security, azure, openai, +3 skills]",No salary
6,Project Manager,https://hirify.me/jobs/440419-project-manager-...,"[hybrid, Poland, fulltime, middle/senior]",Company hidden,8 minutes ago,"[agile, software development, english, waterfa...",No salary
7,Associate Copy Editor,https://hirify.me/jobs/440418-associate-copy-e...,"[US, fulltime]",Company hidden,9 minutes ago,"[agile, grammar, proofreading, punctuation, sp...",43 984 - 105 600\n$
8,Employment Relations Lawyer (Legaltech),https://hirify.me/jobs/440417-employment-relat...,"[hybrid, Sweden, fulltime, middle/senior]",Company hidden,10 minutes ago,"[hr, swedish, employment law, nordic, collecti...",No salary
9,Java Developer,https://hirify.me/jobs/440416-java-developer,"[remote, Mexico, fulltime, middle/senior]",Company hidden,10 minutes ago,"[java, spring, angular, hibernate, soap, +3 sk...",No salary
